# Лекция: Графическое решение задач линейного программирования

**Дисциплина:** Введение в анализ больших данных

**Линейное программирование (ЛП)** — оптимизация линейной целевой функции при линейных ограничениях.

Для **двух переменных** задачу решают **графически**:
1. построить область допустимых решений;
2. нарисовать линии уровня целевой функции;
3. сдвигать линию уровня в направлении градиента до крайней вершины.

Инструменты: **matplotlib**, **scipy.optimize.linprog**.

Числа в примерах **не совпадают** с лабораторным заданием — его решите самостоятельно.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
from scipy.optimize import linprog

plt.rcParams["figure.figsize"] = (9, 7)
print("Библиотеки загружены")


---
## 1. Прямоугольная допустимая область

$$
\max / \min\; c_1 x_1 + c_2 x_2
\quad\text{при}\quad
\ell_1 \le x_1 \le u_1,\;
\ell_2 \le x_2 \le u_2.
$$

Экстремум всегда в одной из вершин прямоугольника.


In [ ]:
c1, c2 = 2, 1
ell1, u1 = 1, 5
ell2, u2 = 0.5, 4
eps = 1.0

fig, ax = plt.subplots()
ax.set_xlim(0, u1 + eps)
ax.set_ylim(0, u2 + eps)
ax.set_xlabel("x1"); ax.set_ylabel("x2")
ax.set_title("ЗЛП: прямоугольная область")

ax.add_patch(Rectangle(
    (ell1, ell2), u1 - ell1, u2 - ell2,
    facecolor="salmon", edgecolor="darkred", alpha=0.55, label="допустимая область",
))

verts = [(ell1, ell2), (ell1, u2), (u1, u2), (u1, ell2)]
for (x, y), lab in zip(verts, "ABCD"):
    ax.plot(x, y, "bo", ms=8)
    ax.text(x + 0.12, y + 0.08, lab, color="blue", fontsize=12)

ax.annotate("", xy=(c1, c2), xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color="blue", lw=2))
ax.text(c1 + 0.1, c2 + 0.1, "grad f", color="blue")

def draw_level(C, ax, style="-"):
    xs = np.linspace(0, u1 + eps, 200)
    ys = (C - c1 * xs) / c2
    ax.plot(xs, ys, style, color="steelblue", alpha=0.65)
    ax.text(xs[-1] * 0.8, (C - c1 * xs[-1] * 0.8) / c2 + 0.05,
            f"{C:.0f}", color="steelblue", fontsize=9)

vals = [c1 * x + c2 * y for x, y in verts]
Cmin, Cmax = min(vals), max(vals)
draw_level(Cmin, ax, "--")
draw_level(Cmax, ax, "--")
for C in np.linspace(0, Cmax + 2, 10):
    draw_level(C, ax)

ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("f в вершинах A,B,C,D:", [round(v, 2) for v in vals])
print(f"Cmin = {Cmin}, Cmax = {Cmax}")


---
## 2. Многоугольник из линейных неравенств

$$
\max\; c^{\top} x
\quad\text{при}\quad
A x \le b,\; x \ge 0.
$$

Вершины — пересечения границ ограничений (с учётом $x \ge 0$).


In [ ]:
c1, c2 = 3, 1
A = np.array([[1.0, 2.0], [3.0, 1.0]])
b = np.array([8.0, 9.0])
x_int = np.linalg.solve(A, b)
print(f"Пересечение границ: ({x_int[0]:.3f}, {x_int[1]:.3f})")

cand = [
    (0.0, 0.0),
    (min(b[0] / A[0, 0], b[1] / A[1, 0]), 0.0),
    tuple(x_int),
    (0.0, min(b[0] / A[0, 1], b[1] / A[1, 1])),
]

def feasible(pt, tol=1e-8):
    x = np.asarray(pt, float)
    return np.all(A @ x <= b + tol) and np.all(x >= -tol)

verts = [v for v in cand if feasible(v)]
print("Вершины:", [(round(x, 3), round(y, 3)) for x, y in verts])


In [ ]:
fig, ax = plt.subplots()
Xmax = max(v[0] for v in verts) + 1.5
Ymax = max(v[1] for v in verts) + 1.5
ax.set_xlim(-0.3, Xmax); ax.set_ylim(-0.3, Ymax)
ax.set_xlabel("x1"); ax.set_ylabel("x2")
ax.set_title("ЗЛП: линейные неравенства")
ax.axhline(0, color="k", lw=1); ax.axvline(0, color="k", lw=1)

xs = np.linspace(0, Xmax, 200)
ax.plot(xs, (b[0] - A[0, 0] * xs) / A[0, 1], "g-", label="огр. 1")
ax.plot(xs, (b[1] - A[1, 0] * xs) / A[1, 1], "m-", label="огр. 2")

order = sorted(verts, key=lambda p: np.arctan2(p[1], p[0]))
ax.add_patch(Polygon(order, closed=True, facecolor="salmon",
                     edgecolor="darkred", lw=2, alpha=0.5, label="область"))

vals = [c1 * x + c2 * y for x, y in verts]
for (x, y), lab, fval in zip(verts, "ABCD", vals):
    ax.plot(x, y, "bo", ms=8)
    ax.text(x + 0.08, y + 0.08, f"{lab}\nf={fval:.2f}", color="blue", fontsize=10)

ax.annotate("", xy=(c1 * 0.7, c2 * 0.7), xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color="blue", lw=2))
ax.text(c1 * 0.7 + 0.05, c2 * 0.7 + 0.05, "grad f", color="blue")

Cmin, Cmax = min(vals), max(vals)
def draw_level(C):
    xs = np.linspace(-0.5, Xmax, 200)
    ax.plot(xs, (C - c1 * xs) / c2, color="steelblue", alpha=0.45, lw=1)
draw_level(Cmin); draw_level(Cmax)
for C in np.linspace(0, Cmax + 1, 7):
    draw_level(C)

imax = int(np.argmax(vals))
print(f"Максимум в {list('ABCD')[imax]} = {verts[imax]}, f={vals[imax]:.3f}")
ax.legend(loc="upper right"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
res = linprog(
    c=[-c1, -c2],
    A_ub=A, b_ub=b,
    bounds=[(0, None), (0, None)],
    method="highs",
)
print("status:", res.message)
print(f"x* = ({res.x[0]:.4f}, {res.x[1]:.4f})")
print(f"max f = {-res.fun:.4f}")


---
## 3. Планирование производства (демо)

Два продукта, два ресурса. Цель — **максимум дохода**.

| Ресурс | Продукт 1 | Продукт 2 | Запас |
|--------|-----------|-----------|-------|
| Материал | 4 | 2 | 24 |
| Труд | 1 | 3 | 15 |
| Доход | 6 | 5 | — |

$$
\max\; 6x_1 + 5x_2
\quad
4x_1 + 2x_2 \le 24,\;
x_1 + 3x_2 \le 15,\;
x \ge 0.
$$


In [ ]:
c = np.array([6.0, 5.0])
A_ub = np.array([[4.0, 2.0], [1.0, 3.0]])
b_ub = np.array([24.0, 15.0])

res = linprog(c=-c, A_ub=A_ub, b_ub=b_ub,
              bounds=[(0, None), (0, None)], method="highs")
print("Оптимальный план:", res.x.round(4))
print("Максимальный доход:", round(-res.fun, 4))

cand = [
    (0.0, 0.0),
    (b_ub[0] / A_ub[0, 0], 0.0),
    tuple(np.linalg.solve(A_ub, b_ub)),
    (0.0, b_ub[1] / A_ub[1, 1]),
]
verts = [v for v in cand if np.all(A_ub @ np.array(v) <= b_ub + 1e-8) and np.all(np.array(v) >= -1e-8)]
verts = sorted(verts, key=lambda p: np.arctan2(p[1], p[0]))

fig, ax = plt.subplots()
xmax = max(v[0] for v in verts) + 2
ymax = max(v[1] for v in verts) + 2
ax.set_xlim(-0.5, xmax); ax.set_ylim(-0.5, ymax)
ax.axhline(0, color="k"); ax.axvline(0, color="k")

xs = np.linspace(0, xmax, 200)
for i in range(A_ub.shape[0]):
    ax.plot(xs, (b_ub[i] - A_ub[i, 0] * xs) / A_ub[i, 1], label=f"огр. {i+1}")

ax.add_patch(Polygon(verts, closed=True, facecolor="lightgreen",
                     edgecolor="darkgreen", lw=2, alpha=0.5))
for v in verts:
    fval = c @ np.array(v)
    ax.plot(*v, "ko", ms=7)
    ax.text(v[0] + 0.1, v[1] + 0.1, f"({v[0]:.1f},{v[1]:.1f})\nf={fval:.1f}", fontsize=9)

opt = res.x
ax.plot(opt[0], opt[1], "r*", ms=16, label=f"optimum f={-res.fun:.2f}")
g = c / np.linalg.norm(c) * 2
ax.annotate("", xy=g, xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color="red", lw=2))
ax.set_xlabel("x1"); ax.set_ylabel("x2")
ax.set_title("Планирование производства")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


### Алгоритм для своей задачи

1. Записать $c_1 x_1 + c_2 x_2 \to \max$ (или min).  
2. Выписать $A x \le b$, $x \ge 0$.  
3. Найти вершины допустимого многоугольника.  
4. Посчитать $f$ в вершинах — экстремум в одной из них.  
5. Проверить `linprog`.  
6. На графике: область, линии уровня, градиент, оптимум.

---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| Прямоугольник | `Rectangle((x,y), w, h)` |
| Многоугольник | `Polygon(verts, ...)` |
| Линия уровня | $c_1 x_1 + c_2 x_2 = C$ → $x_2 = (C - c_1 x_1)/c_2$ |
| Численное решение | `linprog(c=-c, A_ub=A, b_ub=b, bounds=..., method="highs")` |
| (для max) | минимизируем $-c$ |

---
## Что сделать после лекции

1. Повторите построение на **других** коэффициентах.  
2. Откройте лабораторное задание и решите свою производственную задачу **самостоятельно**.  
3. Для $n > 2$ используйте только `linprog`.

Удачи!
